In [ ]:
from gensim.models import FastText

# 샘플 문장 데이터
sentences = [
    ("이커머스 데이터 분석을 진행합니다"),
    ("상품 리뷰를 기반으로 감성 분석을 합니다"),
    ("FastText는 형태소 단위까지 임베딩이 가능합니다"),
]

# FastText 모델 학습
model = FastText(sentences, vector_size=50, window=3, min_count=1, sg=1, epochs=10)

# 특정 단어 벡터 확인
print(model.wv['이커머스'])

# 유사 단어 확인
print(model.wv.most_similar("데이터", topn=5))


[ 2.6210540e-03  9.4743635e-05  3.1530638e-03 -1.2791971e-03
  9.6327747e-04 -6.0999906e-03 -8.2895963e-04  4.2184428e-03
  2.1295536e-04 -3.7958913e-03 -4.8697363e-03 -2.6065344e-03
 -4.6404655e-04 -2.8256543e-03 -2.9324587e-03 -1.4989127e-03
 -1.0608569e-03  9.2257475e-03 -3.6235868e-03  2.3400821e-03
 -1.5622980e-03  2.7761422e-03 -3.2030880e-03  2.2015464e-03
 -5.5274861e-03  7.0813729e-04  1.7380774e-03 -5.7546888e-03
  2.2410129e-03 -6.7082561e-05 -8.5429975e-04 -3.9696745e-03
 -2.7920038e-03 -2.1044204e-03  2.1978309e-03 -1.1105055e-03
  3.8733103e-03  9.9323606e-03  2.0831875e-03  5.0709676e-04
  3.5301242e-03 -2.3441664e-03  1.1695761e-03 -2.7652688e-03
  3.7284836e-03 -1.4906423e-03 -3.1980923e-03 -2.3611574e-04
  5.7185459e-04  5.4654814e-03]
[('성', 0.3870949149131775), ('능', 0.2952795922756195), ('품', 0.26914796233177185), ('형', 0.23869943618774414), ('분', 0.22509989142417908)]


In [15]:
# 학습에 없는 단어
print("없는 단어 벡터 예시:", model.wv["분석가능"])


없는 단어 벡터 예시: [-0.00328122 -0.00349562 -0.00177138  0.00029769 -0.00301265  0.0038021
  0.0001112   0.0051618  -0.00022326 -0.0060964   0.00419311 -0.00273889
 -0.00760797  0.00247095 -0.00384431  0.00185097 -0.00025339  0.00123863
 -0.00200263  0.00194175 -0.00377376  0.00157232  0.00479318 -0.00186722
 -0.00080383  0.00311343  0.00173324  0.00446024 -0.00202898 -0.00144206
  0.00402377  0.0033295   0.00165434 -0.00294058  0.00083202  0.00150245
  0.00619589  0.00169549 -0.00022401  0.00544343  0.0045478  -0.00265199
  0.00594566 -0.00472     0.0089271   0.0026394   0.00307879  0.0033509
 -0.00307856 -0.00282315]


In [32]:
# pip install gensim scikit-learn
# (선택) pip install konlpy jpype1  # Komoran 사용 시
from typing import Callable, List, Optional, Sequence
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression

from gensim.models import FastText

# ---------------------------------------
# 0) 토크나이저: Komoran 우선, 없으면 공백 분할
# ---------------------------------------
def get_tokenizer() -> Callable[[str], List[str]]:
    try:
        from konlpy.tag import Komoran
        komoran = Komoran()
        return lambda s: [t for t in komoran.morphs(s) if t.strip()]
    except Exception:
        # 폴백: 공백 기준 단순 분할
        return lambda s: [t for t in s.split() if t.strip()]

tokenize = get_tokenizer()

# ---------------------------------------
# 1) 커스텀 Transformer: FastTextVectorizer
#    - fit(): FastText 학습
#    - transform(): 문장 벡터(평균 임베딩) 생성
# ---------------------------------------
class FastTextVectorizer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        tokenizer = None,
        vector_size: int = 100,
        window: int = 5,
        min_count: int = 1,
        sg: int = 1,           # 1: skip-gram, 0: CBOW
        epochs: int = 5,
        min_n: int = 3,        # subword 최소 n
        max_n: int = 6,        # subword 최대 n
        workers: int = 1,      # 재현성 위해 1 권장
        seed: int = 42
    ):
        self.tokenizer = tokenizer
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.sg = sg
        self.epochs = epochs
        self.min_n = min_n
        self.max_n = max_n
        self.workers = workers
        self.seed = seed

        self.model_= None
        self.vocab_ = None

    def _to_tokens(self, X: Sequence[str]) -> List[List[str]]:
        if self.tokenizer is None:
            return [[t for t in s.split() if t.strip()] for s in X]
        return [self.tokenizer(s) for s in X]

    def fit(self, X: Sequence[str], y=None):
        sentences = self._to_tokens(X)
        # FastText 학습 (fold마다 새로 학습되어 CV 누수 방지)
        self.model_ = FastText(
            sentences=sentences,
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            sg=self.sg,
            epochs=self.epochs,
            min_n=self.min_n,
            max_n=self.max_n,
            workers=self.workers,
            seed=self.seed,
        )
        self.vocab_ = set(self.model_.wv.key_to_index.keys())
        return self
    # 한 문장의 토큰 리스트를 하나의 평균 벡터로 변환
    # pipeline에서 사용하는 내부 함수 
    def _doc_vec(self, tokens: List[str]) -> np.ndarray:
        if self.model_ is None:
            raise RuntimeError("Model is not fitted yet.")
        vecs = [self.model_.wv[w] for w in tokens if w in self.model_.wv]
        if not vecs:
            v = np.zeros(self.model_.vector_size, dtype=np.float32)
        else:
            v = np.mean(vecs, axis=0).astype(np.float32)
        return v

    def transform(self, X: Sequence[str]) -> np.ndarray:
        sentences = self._to_tokens(X)
        mat = np.vstack([self._doc_vec(tokens) for tokens in sentences])
        return mat

# ---------------------------------------
# 2) 데이터 (데모용 소규모 한글 문장)
#    - 실제로는 NSMC 등으로 교체하여 사용하세요.
# ---------------------------------------
X = [
    "상품 품질이 아주 좋아요",
    "배송이 너무 느립니다",
    "포장이 깔끔하고 만족합니다",
    "환불이 지연되어 불만입니다",
    "가격 대비 만족스러워요",
    "연락이 안 되고 불친절했어요",
    "디자인이 예쁘고 마음에 들어요",
    "설명과 달라서 실망입니다",
    "배송 빠르고 기사님 친절했어요",
    "색상이 사진과 같아 만족합니다",
]
y = [1,0,1,0,1,0,1,0,1,1]  # 1=긍정, 0=부정 (데모 라벨)

# ---------------------------------------
# 3) 파이프라인 & GridSearchCV 구성
# ---------------------------------------
pipe = Pipeline([
    ("ft", FastTextVectorizer(tokenizer=tokenize)),
    ("clf", LogisticRegression(max_iter=2000))
])

param_grid = {
    # FastText 하이퍼파라미터
    "ft__vector_size": [50, 100],
    # "ft__window": [3, 5],
    "ft__sg": [0, 1],             # CBOW vs Skip-gram
    # "ft__epochs": [5, 10],

    # 분류기(LogReg) 하이퍼파라미터
    "clf__C": [1.0, 3.0]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

gs = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1_macro",   # 불균형이면 f1_weighted 등 고려
    cv=cv,
    verbose=1
)

gs.fit(X, y)

print("★ Best Params:", gs.best_params_)
print("★ Best CV F1:", gs.best_score_)

# ---------------------------------------
# 4) 최종 평가 (데모에선 CV로 충분 / 실제는 별도 test 권장)
#    - 여기서는 간단히 best_estimator_로 다시 예측
# ---------------------------------------
best_pipe = gs.best_estimator_
pred = best_pipe.predict(X)
print("\n[Classification Report on the toy data]")
print(classification_report(y, pred, digits=4))


Fitting 3 folds for each of 8 candidates, totalling 24 fits
★ Best Params: {'clf__C': 1.0, 'ft__sg': 0, 'ft__vector_size': 50}
★ Best CV F1: 0.37777777777777777

[Classification Report on the toy data]
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         4
           1     0.6000    1.0000    0.7500         6

    accuracy                         0.6000        10
   macro avg     0.3000    0.5000    0.3750        10
weighted avg     0.3600    0.6000    0.4500        10



c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [ ]:
# !pip install optuna

In [33]:
# pip install optuna gensim scikit-learn
# (선택) pip install konlpy jpype1  # Komoran 사용 시

import numpy as np
import optuna
from typing import List, Callable, Sequence
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from gensim.models import FastText

# -------------------------------------------------
# 0) 토크나이저: Komoran 우선, 없으면 공백 분할
# -------------------------------------------------
def get_tokenizer() -> Callable[[str], List[str]]:
    try:
        from konlpy.tag import Komoran
        komoran = Komoran()
        return lambda s: [t for t in komoran.morphs(s) if t.strip()]
    except Exception:
        return lambda s: [t for t in s.split() if t.strip()]

tok = get_tokenizer()

# -------------------------------------------------
# 1) 데모 데이터 (실전에서는 NSMC 등으로 교체)
# -------------------------------------------------
X_text = [
    "상품 품질이 아주 좋아요",
    "배송이 너무 느립니다",
    "포장이 깔끔하고 만족합니다",
    "환불이 지연되어 불만입니다",
    "가격 대비 만족스러워요",
    "연락이 안 되고 불친절했어요",
    "디자인이 예쁘고 마음에 들어요",
    "설명과 달라서 실망입니다",
    "배송 빠르고 기사님 친절했어요",
    "색상이 사진과 같아 만족합니다",
]
y = np.array([1,0,1,0,1,0,1,0,1,1])  # 1=긍정, 0=부정

X_tokens = [tok(s) for s in X_text]

# -------------------------------------------------
# 2) 문장 → 평균 임베딩 벡터
# -------------------------------------------------
def sent_vec(ft: FastText, tokens: Sequence[str], normalize=True) -> np.ndarray:
    vecs = [ft.wv[w] for w in tokens if w in ft.wv]
    if not vecs:
        v = np.zeros(ft.vector_size, dtype=np.float32)
    else:
        v = np.mean(vecs, axis=0).astype(np.float32)
    return v

# -------------------------------------------------
# 3) Optuna 목적 함수 (교차검증 포함)
#    - 각 trial에서 FastText 학습 + 문장벡터화 + 분류기 학습/평가
# -------------------------------------------------
def objective(trial: optuna.trial.Trial) -> float:
    # FastText 하이퍼파라미터 탐색 공간
    vector_size = trial.suggest_categorical("vector_size", [50, 100, 200])
    window      = trial.suggest_int("window", 3, 7)
    min_count   = trial.suggest_int("min_count", 1, 2)
    sg          = trial.suggest_categorical("sg", [0, 1])      # 0=CBOW, 1=Skip-gram
    epochs      = trial.suggest_int("epochs", 5, 15)
    min_n       = trial.suggest_int("min_n", 3, 4)             # subword
    max_n       = trial.suggest_int("max_n", 5, 6)

    # 분류기 선택 및 하이퍼파라미터
    clf_type    = trial.suggest_categorical("clf_type", ["logreg", "linear_svm"])
    if clf_type == "logreg":
        C = trial.suggest_float("C", 1e-3, 10.0, log=True)
        clf_ctor = lambda: LogisticRegression(C=C, max_iter=2000)
    else:
        C = trial.suggest_float("C", 1e-3, 10.0, log=True)
        clf_ctor = lambda: LinearSVC(C=C)

    # Stratified K-Fold CV
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tokens, y), 1):
        tr_tokens = [X_tokens[i] for i in tr_idx]
        va_tokens = [X_tokens[i] for i in va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        # 폴드의 train 데이터로만 FastText 학습 (데이터 누수 방지)
        ft = FastText(
            sentences=tr_tokens,
            vector_size=vector_size,
            window=window,
            min_count=min_count,
            sg=sg,
            epochs=epochs,
            min_n=min_n,
            max_n=max_n,
            workers=1,   # 재현성
            seed=42
        )

        # 문장 벡터화
        X_tr = np.vstack([sent_vec(ft, t) for t in tr_tokens])
        X_va = np.vstack([sent_vec(ft, t) for t in va_tokens])

        # 분류기 학습 & 평가
        clf = clf_ctor()
        clf.fit(X_tr, y_tr)
        pred = clf.predict(X_va)
        f1 = f1_score(y_va, pred, average="macro")
        scores.append(f1)

        # 중간 리포트 & 프루닝
        trial.report(f1, step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(scores))

# -------------------------------------------------
# 4) 최적화 실행
# -------------------------------------------------
study = optuna.create_study(direction="maximize", study_name="ft_optuna_demo")
study.optimize(objective, n_trials=20, show_progress_bar=True)

print("Best params:", study.best_params)
print("Best CV F1:", study.best_value)

# -------------------------------------------------
# 5) 최적 파라미터로 전체 데이터 재학습 & 리포트
# -------------------------------------------------
bp = study.best_params

ft_final = FastText(
    sentences=X_tokens,
    vector_size=bp.get("vector_size", 100),
    window=bp.get("window", 5),
    min_count=bp.get("min_count", 1),
    sg=bp.get("sg", 1),
    epochs=bp.get("epochs", 10),
    min_n=bp.get("min_n", 3),
    max_n=bp.get("max_n", 6),
    workers=1,
    seed=42
)

X_all = [sent_vec(ft_final, t) for t in X_tokens]

if bp["clf_type"] == "logreg":
    clf_final = LogisticRegression(C=bp["C"], max_iter=2000)
else:
    clf_final = LinearSVC(C=bp["C"])

clf_final.fit(X_all, y)
pred_all = clf_final.predict(X_all)
print("\n[Classification Report on the toy data]")
print(classification_report(y, pred_all, digits=4))


[I 2025-10-31 01:44:06,987] A new study created in memory with name: ft_optuna_demo
Best trial: 0. Best value: 0.377778:   5%|▌         | 1/20 [00:02<00:53,  2.83s/it]

[I 2025-10-31 01:44:09,819] Trial 0 finished with value: 0.37777777777777777 and parameters: {'vector_size': 200, 'window': 5, 'min_count': 1, 'sg': 1, 'epochs': 7, 'min_n': 4, 'max_n': 5, 'clf_type': 'logreg', 'C': 6.304690767201185}. Best is trial 0 with value: 0.37777777777777777.


Best trial: 0. Best value: 0.377778:  10%|█         | 2/20 [00:04<00:36,  2.00s/it]

[I 2025-10-31 01:44:11,237] Trial 1 finished with value: 0.37777777777777777 and parameters: {'vector_size': 100, 'window': 6, 'min_count': 2, 'sg': 1, 'epochs': 13, 'min_n': 3, 'max_n': 5, 'clf_type': 'logreg', 'C': 1.149160699692527}. Best is trial 0 with value: 0.37777777777777777.


Best trial: 0. Best value: 0.377778:  15%|█▌        | 3/20 [00:07<00:42,  2.50s/it]

...

Best trial: 0. Best value: 0.377778:  95%|█████████▌| 19/20 [00:36<00:01,  1.83s/it]

[I 2025-10-31 01:44:43,417] Trial 18 finished with value: 0.37777777777777777 and parameters: {'vector_size': 100, 'window': 5, 'min_count': 2, 'sg': 1, 'epochs': 7, 'min_n': 4, 'max_n': 5, 'clf_type': 'linear_svm', 'C': 0.07362425780826284}. Best is trial 0 with value: 0.37777777777777777.


Best trial: 0. Best value: 0.377778: 100%|██████████| 20/20 [00:37<00:00,  1.86s/it]


[I 2025-10-31 01:44:44,136] Trial 19 finished with value: 0.37777777777777777 and parameters: {'vector_size': 50, 'window': 5, 'min_count': 1, 'sg': 1, 'epochs': 6, 'min_n': 3, 'max_n': 5, 'clf_type': 'logreg', 'C': 4.426353174455394}. Best is trial 0 with value: 0.37777777777777777.
Best params: {'vector_size': 200, 'window': 5, 'min_count': 1, 'sg': 1, 'epochs': 7, 'min_n': 4, 'max_n': 5, 'clf_type': 'logreg', 'C': 6.304690767201185}
Best CV F1: 0.37777777777777777

[Classification Report on the toy data]
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         4
           1     0.6000    1.0000    0.7500         6

    accuracy                         0.6000        10
   macro avg     0.3000    0.5000    0.3750        10
weighted avg     0.3600    0.6000    0.4500        10



c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [34]:

import numpy as np
import optuna
from typing import List, Sequence
from konlpy.tag import Komoran
from gensim.models import Word2Vec
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# -----------------------------
# A. 데이터 로드 (NSMC: 네이버 영화 리뷰)
# -----------------------------
# 'document' (텍스트), 'label' (0/1)
ds = pd.read_csv("data/ratings_train.txt", sep='\t')
train_texts = ds["document"].head(500)
train_labels = np.array(ds["label"].head(500))
test_texts  = ds["document"].tail(30)
test_labels = np.array(ds["label"].tail(30))

# 간단 전처리(선택): 이모지/특수문자 최소화, 공백 정리
def clean_text(s: str) -> str:
    if s is None: 
        return ""
    s = s.strip()
    return s

train_texts = [clean_text(t) for t in train_texts]
test_texts  = [clean_text(t) for t in test_texts]

# -----------------------------
# B. Komoran 토큰화
# -----------------------------
komoran = Komoran()

def tokenize_komoran(text: str) -> List[str]:
    # 불용어/품사 필터가 필요하면 여기서 적용 (예: 조사/어미 제거 등)
    # ex) pos = komoran.pos(text);  특정 품사만 남기는 식으로 커스터마이즈 가능
    return [tok for tok in komoran.morphs(text) if tok and tok.strip()]

train_tokens: List = [tokenize_komoran(t) for t in train_texts]
test_tokens:  List = [tokenize_komoran(t) for t in test_texts]

# -----------------------------
# C. Optuna 목적 클래스 (trial을 멤버로 포함)
# -----------------------------
class W2VClassifierObjective:
    """
    - trial을 멤버로 포함하여 W2V 하이퍼파라미터 + 분류기 하이퍼파라미터 동시 탐색
    - StratifiedKFold CV에서 각 폴드별 학습/평가 (데이터 누수 방지)
    - 문장 임베딩은 평균(선택적으로 L2 정규화)
    """
    def __init__(
        self,
        sentences: Sequence[Sequence[str]],
        y: np.ndarray,
        n_splits: int = 3,
        normalize: bool = True,
        use_tfidf_weight: bool = False,
        random_state: int = 42,
    ):
        self.sentences = list(sentences)
        self.y = np.asarray(y)
        self.n_splits = n_splits
        self.normalize = normalize
        self.use_tfidf_weight = use_tfidf_weight
        self.random_state = random_state
        self.trial = None

    def __call__(self, trial: optuna.trial.Trial) -> float:
        self.trial = trial
        return self.cross_val_score()

    def fit_w2v(self, train_sentences: Sequence[Sequence[str]]):
        t = self.trial
        vector_size = t.suggest_categorical("w2v_vector_size", [100, 200, 300])
        window      = t.suggest_int("w2v_window", 3, 7)
        min_count   = t.suggest_int("w2v_min_count", 1, 3)
        sg          = t.suggest_categorical("w2v_sg", [0, 1])   # 0: CBOW, 1: skip-gram
        epochs      = t.suggest_int("w2v_epochs", 5, 10)

        model = Word2Vec(
            sentences=train_sentences,
            vector_size=vector_size,
            window=window,
            min_count=min_count,
            sg=sg,
            epochs=epochs,
            workers=1,
            seed=self.random_state,
        )
        return model

    def sentence_vector(self, model: Word2Vec, tokens: Sequence[str]) -> np.ndarray:
        vecs = [model.wv[w] for w in tokens if w in model.wv]
        if len(vecs) == 0:
            v = np.zeros(model.vector_size, dtype=np.float32)
        else:
            v = np.mean(vecs, axis=0)
        if self.normalize:
            n = np.linalg.norm(v) + 1e-12
            v = v / n
        return v.astype(np.float32)

    def build_classifier(self):
        t = self.trial
        clf_type = t.suggest_categorical("clf_type", ["logreg", "linear_svm", "rbf_svm", "rf"])
        if clf_type == "logreg":
            C = t.suggest_float("logreg_C", 1e-3, 10.0, log=True)
            return LogisticRegression(C=C, max_iter=2000)
        elif clf_type == "linear_svm":
            C = t.suggest_float("linSVM_C", 1e-3, 10.0, log=True)
            return LinearSVC(C=C)
        elif clf_type == "rbf_svm":
            C = t.suggest_float("rbfSVM_C", 1e-2, 50.0, log=True)
            gamma = t.suggest_float("rbfSVM_gamma", 1e-4, 1.0, log=True)
            return SVC(C=C, gamma=gamma, kernel="rbf")
        else:
            n_estimators = t.suggest_int("rf_n_estimators", 100, 400)
            max_depth    = t.suggest_int("rf_max_depth", 5, 25)
            min_split    = t.suggest_int("rf_min_samples_split", 2, 10)
            min_leaf     = t.suggest_int("rf_min_samples_leaf", 1, 5)
            return RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_split=min_split,
                min_samples_leaf=min_leaf,
                random_state=self.random_state
            )

    def cross_val_score(self) -> float:
        skf = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        scores = []

        for fold, (tr_idx, va_idx) in enumerate(skf.split(self.sentences, self.y), 1):
            s_train = [self.sentences[i] for i in tr_idx]
            s_valid = [self.sentences[i] for i in va_idx]
            y_train, y_valid = self.y[tr_idx], self.y[va_idx]

            w2v = self.fit_w2v(s_train)

            X_train = np.vstack([self.sentence_vector(w2v, s) for s in s_train])
            X_valid = np.vstack([self.sentence_vector(w2v, s) for s in s_valid])

            clf = self.build_classifier()
            clf.fit(X_train, y_train)
            pred = clf.predict(X_valid)

            f1 = f1_score(y_valid, pred, average="macro")
            scores.append(f1)

            # Optuna 중간 리포트 & 프루닝
            self.trial.report(f1, step=fold)
            if self.trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(scores))

# -----------------------------
# D. Optuna 최적화 실행 (학습 시간 고려해 trial 수는 처음엔 작게)
# -----------------------------
objective = W2VClassifierObjective(
    sentences=train_tokens,
    y=train_labels,
    n_splits=3,
    normalize=True,
    use_tfidf_weight=False,   # 필요하면 True로 바꾸고 TF-IDF 가중 평균 로직 추가 가능
    random_state=42
)

study = optuna.create_study(direction="maximize", study_name="nsmc_komoran_w2v_ml")
study.optimize(objective, n_trials=20, show_progress_bar=True)

print("Best params:", study.best_params)
print("Best CV F1:", study.best_value)

# -----------------------------
# E. 테스트 세트 최종 평가 (Best 파라미터로 재학습)
# -----------------------------
best = study.best_params

# 1) Word2Vec 재학습 (train 전체)
w2v_final = Word2Vec(
    sentences=train_tokens,
    vector_size=best.get("w2v_vector_size", 200),
    window=best.get("w2v_window", 5),
    min_count=best.get("w2v_min_count", 1),
    sg=best.get("w2v_sg", 1),
    epochs=best.get("w2v_epochs", 10),
    workers=1,
    seed=42
)

def sentvec(model, toks):
    vs = [model.wv[w] for w in toks if w in model.wv]
    v = np.mean(vs, axis=0) if vs else np.zeros(model.vector_size, dtype=np.float32)
    v = v / (np.linalg.norm(v) + 1e-12)
    return v.astype(np.float32)

X_train = [sentvec(w2v_final, s) for s in train_tokens]
X_test  = [sentvec(w2v_final, s) for s in test_tokens]

# 2) 분류기 구성
clf_type = best["clf_type"]
if clf_type == "logreg":
    clf = LogisticRegression(C=best["logreg_C"], max_iter=2000)
elif clf_type == "linear_svm":
    clf = LinearSVC(C=best["linSVM_C"])
elif clf_type == "rbf_svm":
    clf = SVC(C=best["rbfSVM_C"], gamma=best["rbfSVM_gamma"], kernel="rbf")
else:
    clf = RandomForestClassifier(
        n_estimators=best["rf_n_estimators"],
        max_depth=best["rf_max_depth"],
        min_samples_split=best["rf_min_samples_split"],
        min_samples_leaf=best["rf_min_samples_leaf"],
        random_state=42
    )

clf.fit(X_train, train_labels)
pred = clf.predict(X_test)
print(classification_report(test_labels, pred, digits=4))


[I 2025-10-31 01:46:36,919] A new study created in memory with name: nsmc_komoran_w2v_ml
Best trial: 0. Best value: 0.340371:  10%|█         | 2/20 [00:00<00:02,  8.47it/s]

[I 2025-10-31 01:46:37,015] Trial 0 finished with value: 0.3403705795010143 and parameters: {'w2v_vector_size': 200, 'w2v_window': 6, 'w2v_min_count': 3, 'w2v_sg': 0, 'w2v_epochs': 8, 'clf_type': 'logreg', 'logreg_C': 0.0029500803561282435}. Best is trial 0 with value: 0.3403705795010143.
[I 2025-10-31 01:46:37,156] Trial 1 finished with value: 0.3403705795010143 and parameters: {'w2v_vector_size': 200, 'w2v_window': 5, 'w2v_min_count': 1, 'w2v_sg': 0, 'w2v_epochs': 8, 'clf_type': 'logreg', 'logreg_C': 0.16255191700963295}. Best is trial 0 with value: 0.3403705795010143.


Best trial: 2. Best value: 0.641217:  20%|██        | 4/20 [00:01<00:04,  3.36it/s]

[I 2025-10-31 01:46:37,932] Trial 2 finished with value: 0.6412171507459948 and parameters: {'w2v_vector_size': 300, 'w2v_window': 4, 'w2v_min_count': 1, 'w2v_sg': 0, 'w2v_epochs': 6, 'clf_type': 'rf', 'rf_n_estimators': 143, 'rf_max_depth': 18, 'rf_min_samples_split': 10, 'rf_min_samples_leaf': 4}. Best is trial 2 with value: 0.6412171507459948.
[I 2025-10-31 01:46:38,069] Trial 3 finished with value: 0.3403705795010143 and parameters: {'w2v_vector_size': 100, 'w2v_window': 7, 'w2v_min_count': 1, 'w2v_sg': 0, 'w2v_epochs': 9, 'clf_type': 'linear_svm', 'linSVM_C': 0.34663346325048316}. Best is trial 2 with value: 0.6412171507459948.


Best trial: 2. Best value: 0.641217:  30%|███       | 6/20 [00:02<00:05,  2.57it/s]

...

[I 2025-10-31 01:46:45,067] Trial 16 finished with value: 0.6675446365808942 and parameters: {'w2v_vector_size': 300, 'w2v_window': 4, 'w2v_min_count': 2, 'w2v_sg': 1, 'w2v_epochs': 6, 'clf_type': 'rf', 'rf_n_estimators': 398, 'rf_max_depth': 10, 'rf_min_samples_split': 4, 'rf_min_samples_leaf': 1}. Best is trial 16 with value: 0.6675446365808942.


Best trial: 16. Best value: 0.667545:  90%|█████████ | 18/20 [00:09<00:01,  1.05it/s]

[I 2025-10-31 01:46:46,758] Trial 17 finished with value: 0.638137722082941 and parameters: {'w2v_vector_size': 200, 'w2v_window': 6, 'w2v_min_count': 3, 'w2v_sg': 1, 'w2v_epochs': 6, 'clf_type': 'rf', 'rf_n_estimators': 371, 'rf_max_depth': 12, 'rf_min_samples_split': 5, 'rf_min_samples_leaf': 2}. Best is trial 16 with value: 0.6675446365808942.


Best trial: 16. Best value: 0.667545: 100%|██████████| 20/20 [00:11<00:00,  1.67it/s]


[I 2025-10-31 01:46:48,818] Trial 18 finished with value: 0.6547501794468616 and parameters: {'w2v_vector_size': 300, 'w2v_window': 3, 'w2v_min_count': 2, 'w2v_sg': 1, 'w2v_epochs': 7, 'clf_type': 'rf', 'rf_n_estimators': 386, 'rf_max_depth': 11, 'rf_min_samples_split': 4, 'rf_min_samples_leaf': 2}. Best is trial 16 with value: 0.6675446365808942.
[I 2025-10-31 01:46:48,864] Trial 19 pruned. 
Best params: {'w2v_vector_size': 300, 'w2v_window': 4, 'w2v_min_count': 2, 'w2v_sg': 1, 'w2v_epochs': 6, 'clf_type': 'rf', 'rf_n_estimators': 398, 'rf_max_depth': 10, 'rf_min_samples_split': 4, 'rf_min_samples_leaf': 1}
Best CV F1: 0.6675446365808942
              precision    recall  f1-score   support

           0     0.5714    0.6154    0.5926        13
           1     0.6875    0.6471    0.6667        17

    accuracy                         0.6333        30
   macro avg     0.6295    0.6312    0.6296        30
weighted avg     0.6372    0.6333    0.6346        30

